In [ ]:
import pandas as pd
import matplotlib as plt
import seaborn as sns
import numpy as np
from pathlib import Path # manage paths for the project

In [ ]:
# define the path of the data
data_path = Path('..') / 'data' / 'raw' / 'educacionCol.csv'

In [ ]:
# Cargar dataset

df = pd.read_csv(data_path, low_memory=False)
df.head()

In [ ]:
df.tail()

In [ ]:
# exploracion inicial
print(f"Dimensiones del dataset: {df.shape[0]} filas y {df.shape[1]} columnas \n")
df.info()

In [ ]:
# Analisis de valores nulos

nulos = df.isnull().sum()
nulos_porcentaje = (nulos / len(df)) * 100
df_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': nulos_porcentaje})
print(df_nulos)

# Mostrar solo las columnas que tienen nulos
display(df_nulos[df_nulos['Nulos'] > 0].sort_values(by='Porcentaje (%)', ascending=False))

# No tenemos nulos

In [ ]:
total_duplicados = df.duplicated().sum()
print(f"Se encontraron {total_duplicados} filas exactamente duplicadas en el dataset.")

if total_duplicados > 0:
    # Mostrar una muestra de los duplicados
    display(df[df.duplicated(keep=False)].sort_values(by='Código SNIES delprograma').head(4))

In [ ]:
# Validacion de tipos de datos
display(df.dtypes)

# Total de matriculados esta como texto hay que corregirlo

In [ ]:
df['Municipio de oferta del programa'].unique()

In [ ]:
# 2. Forzar conversión a numérico, rellenar nulos con 0 y pasar a entero
df['Total Matriculados'] = pd.to_numeric(df['Total Matriculados'], errors='coerce').fillna(0).astype('int')

# --- Gráfico de Outliers ---
plt.figure(figsize=(10, 2))
sns.boxplot(x=df['Total Matriculados'])
plt.title('Distribución de Total Matriculados (Búsqueda de Outliers)')
plt.show()

# --- Cuantificación de Outliers (Método IQR) ---
Q1 = df['Total Matriculados'].quantile(0.25)
Q3 = df['Total Matriculados'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR
outliers = df[df['Total Matriculados'] > limite_superior]
print(f"Total de Outliers: {len(outliers)}")

In [16]:
# Revisar si hay municipios escritos de diferentes formas (Mayúsculas vs Minúsculas o espacios al final)
municipios_unicos = df['Municipio de oferta del programa'].dropna().unique()
print(f"Total municipios únicos (sin limpiar): {len(municipios_unicos)}")

# Aplicando strip y upper para ver si el número se reduce (lo que indicaría inconsistencias de formato)
municipios_limpios = df['Municipio de oferta del programa'].dropna().astype(str).str.strip().str.upper().unique()
print(f"Total municipios únicos (limpiando espacios y mayúsculas): {len(municipios_limpios)}")

# Verificar longitudes raras en los años (deberían ser 4 caracteres)
años_inconsistentes = df[df['Año'].astype(str).str.len() != 4]['Año'].unique()
if len(años_inconsistentes) > 0:
    print(f"Años con formato inconsistente encontrados: {años_inconsistentes}")
else:
    print("La columna 'Año' tiene un formato consistente (4 dígitos).")

Total municipios únicos (sin limpiar): 1439
Total municipios únicos (limpiando espacios y mayúsculas): 1004
La columna 'Año' tiene un formato consistente (4 dígitos).
